# Training the U-Net (issues #9, #10)

U-Net with a ResNet-34 encoder pretrained on ImageNet, reflection padding,
loss BCE + batch Dice. Checkpoints go to Drive: if the session is cut, run
the cells again and training resumes from `last.pt`.

**Runtime: GPU** (Runtime > Change runtime type > T4 GPU).

| Cell | What it does |
|---|---|
| 1 | Parameters (your values) |
| 2 | Mount Drive, get the code, install the packages |
| 3 | Check the GPU |
| 4 | Copy the dataset from Drive to the VM's disk |
| 5 | Train (or resume) |
| 6 | Training curves |

In [ ]:
REPO_URL = "https://github.com/bonlieuguillaume/aletheia.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/aletheia"

DATASET_ZIP = DRIVE_ROOT + "/data/dataset_v0.1_level1.zip"
LOCAL_DATASET = "/content/dataset"
RUN_NAME = "v0.1_level1_resnet34"
RUN_DIR = DRIVE_ROOT + "/runs/" + RUN_NAME

# Overrides of unet.DEFAULT_CONFIG (only used when the run starts, not on resume).
CONFIG = {
    "encoder": "resnet34",
    "pretrained": True,
    "input_scale": "linear",   # "db" if the scenes are already in dB (see the SNAP XML)
    "augment": False,          # True: east-west mirror half of the time (#37: compare both)
    "batch_size": 8,
    "epochs": 40,
    "learning_rate": 3e-4,
}

In [ ]:
# Mount Google Drive, get the latest code from GitHub, install the packages.
# Everything happens on the Colab VM: nothing is installed on your PC.
import os
import subprocess

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/aletheia"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run(["pip", "install", "-q", "-r", REPO_DIR + "/requirements.txt"], check=True)

# The notebook runs from its own folder and imports the module next to it.
os.chdir(REPO_DIR + "/features/unet")
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "none (check the runtime type)")

In [ ]:
import os

from features.sar_io.sar_io import unzip_folder

if not os.path.exists(LOCAL_DATASET + "/manifest.csv"):
    unzip_folder(DATASET_ZIP, LOCAL_DATASET)

In [ ]:
from unet import train

best_checkpoint = train(LOCAL_DATASET, RUN_DIR, CONFIG)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

log = pd.read_csv(RUN_DIR + "/log.csv")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(log["epoch"], log["train_loss"], label="train")
axes[0].plot(log["epoch"], log["val_loss"], label="val")
axes[0].set_title("Loss (BCE + Dice)")
axes[0].legend()
axes[1].plot(log["epoch"], log["val_image_mcc_at_0.5"], label="image MCC")
axes[1].plot(log["epoch"], log["val_pixel_mcc_at_0.5"], label="pixel MCC")
axes[1].set_title("Validation, threshold 0.5 (monitoring only)")
axes[1].legend()
plt.show()
log.tail()